# <span style="color:#5E6997"> Technical Indicators Bitcoin Market Data </span>

## <span style="color:#5E6997"> Leveraging bitcoin' minutes market data </span>

<br>

### Data Manipulation

- Merging market data all years
- Handling upper/lower cases differences and features spellings
- Dropping extra columns ('unix')
- Features formatting
- Feature engineering 

<br>

### Features Engineering

**Features creation**
- token using 'symbol'
- 'day' and 'hour' using 'date'

**Technical Indicators**

- **Multiple EMAs**: *Moving average that gives more weight to recent data points, making it more responsive to current price movements.*
- **WMA**: *14-period WMA, moving average that assigns different weights to each data point in the series.*
- **MACD**: *MACD, Signal Line, and Histogram, a trend-following momentum indicator that shows the relationship between two moving averages.*
- **ATR**: *Average True Range, measures market volatility by calculating the average range between high and low prices.*
- **HMA**: *Hull Moving Average, weighted moving average that aims to reduce lag and increase responsiveness to price changes.*
- **KAMA**: *Kaufman's Adaptive Moving Average, adaptive moving average that adjusts its speed based on market conditions.*
- **CMO**: *Chande Momentum Oscillator, Measures price changes relative to the total range of prices over a specified period.*
- **Z-Score**: *Z-Score, Measures how far a data point is from the mean in terms of standard deviations.*
- **QStick**: *QStick, Measures momentum based on the difference between the opening and closing prices.*

Using the library PandasTA.


Enjoy!

In [ ]:
pip install pandas-ta

In [ ]:
import numpy as np
import pandas as pd
import os
from pandas_ta import wma, macd, stoch, adx, atr, hma, kama, cmo, zscore, qstick

In [ ]:
# df_2021 = pd.read_csv('/kaggle/input/bitcoin-minute-data/Binance_BTCUSDT_2021_minute.csv')
# df_2021 = df_2021.drop(columns=['marketorder_volume', 'marketorder_volume_from','date_close', 'close_unix'])
# df_2021.to_csv('BTC_2021.csv', index=False)

# Importing, Handling & Merging all datasets 
(due to size and iteration limits, data had to be splitted by year. Roughly 525600 minutes in a year. So now we are merging it back)

In [ ]:
directory = '/kaggle/input/bitcoin-minute-data'

file_names = [
    'Binance_BTCUSDT_2020_minute.csv',
    'BTC_2021.csv',
    'Binance_BTCUSDT_2022_minute.csv',
    'Binance_BTCUSDT_2023_minute.csv',
]

df_crypto = pd.DataFrame()

for file_name in file_names:
    file_path = os.path.join(directory, file_name)
    df = pd.read_csv(file_path)

    if file_name == 'Binance_BTCUSDT_2023_minute.csv':
        df.columns = map(str.lower, df.columns)

    if file_name == 'BTC_2021.csv':
        df = df.rename(columns={'volume': 'Volume BTC', 'volume_from': 'Volume USDT'})

    if file_name == 'Binance_BTCUSDT_2022_minute.csv':
        df = df.rename(columns={'volume': 'Volume BTC', 'volume_from': 'Volume USDT'})
        
    if file_name == 'Binance_BTCUSDT_2023_minute.csv':
        df = df.rename(columns={'volume usdt': 'Volume USDT'})

    df = df.drop(columns=df.columns[7])

    df_crypto = pd.concat([df_crypto, df], ignore_index=True)

In [ ]:
df_crypto.head()

In [ ]:
df = df_crypto.copy(deep=True)
df = df.drop(columns=['unix'])

In [ ]:
df.head()

In [ ]:
df['token'] = df['symbol'].str.split('USDT').str[0]
df['token'] = df['token'].str.replace('/', '', regex=False)

In [ ]:
df.info()

# Formatting 'Volume USDT' and 'date'

In [ ]:
df['Volume USDT'] = df['Volume USDT'].astype(float).round(0).astype(int)
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d %H:%M:%S')

# Creating Features 'day' and 'hour'

In [ ]:
df['hour'] = df['date'].dt.hour
df['day'] = df['date'].dt.day_name()

# Adding EMA to the dataframe

In [ ]:
df['ema_5'] = df['close'].ewm(span=5).mean()
df['ema_15'] = df['close'].ewm(span=15).mean()
df['ema_30'] = df['close'].ewm(span=30).mean()
df['ema_60'] = df['close'].ewm(span=60).mean()
df['ema_100'] = df['close'].ewm(span=100).mean()
df['ema_200'] = df['close'].ewm(span=200).mean()

In [ ]:
df.head()

# Adding Technical Indicators to our dataframe

- **WMA**: *14-period WMA, moving average that assigns different weights to each data point in the series.*
- **MACD**: *MACD, Signal Line, and Histogram, a trend-following momentum indicator that shows the relationship between two moving averages.*
- **ATR**: *Average True Range, measures market volatility by calculating the average range between high and low prices.*
- **HMA**: *Hull Moving Average, weighted moving average that aims to reduce lag and increase responsiveness to price changes.*
- **KAMA**: *Kaufman's Adaptive Moving Average, adaptive moving average that adjusts its speed based on market conditions.*
- **CMO**: *Chande Momentum Oscillator, Measures price changes relative to the total range of prices over a specified period.*
- **Z-Score**: *Z-Score, Measures how far a data point is from the mean in terms of standard deviations.*
- **QStick**: *QStick, Measures momentum based on the difference between the opening and closing prices.*

library: Pandas TA

**Note**: Obviously technical indicators computing average, change, range, etc.. will display NaN until n rows.

In [ ]:
df['WMA'] = wma(df['close'], length=14)
df[['MACD', 'MACD_Signal', 'MACD_Hist']] = macd(df['close'])
df['ATR'] = atr(df['high'], df['low'], df['close'])
df['HMA'] = hma(df['close'])
df['KAMA'] = kama(df['close'])
df['CMO'] = cmo(df['close'])
df['Z-Score'] = zscore(df['close'])
df['QStick'] = qstick(df['open'], df['close'])

In [ ]:
df.head(2)

In [ ]:
df.sample(2)

In [ ]:
df.tail(2)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
df = df.sort_values(by='date', ascending=True)

In [ ]:
df.to_csv('power_data.csv', index=False)